# jobs: FashionMNIST

project = ```iP-VAE```, host = ```yoru/mach```, device = ```any```

**Motivation**: <br>

Create jobs for all the EMNIST dataset fits.

- poisson: (128, 128.0)
- gaussian: (64, 64.0)
- gaussian+relu: (16, 16.0)

In [1]:
# HIDE CODE


import os, sys
from IPython.display import display

project_name = '_IterativeVAE'

# tmp & extras dir
git_dir = os.path.join(os.environ['HOME'], 'Dropbox/git')
extras_dir = os.path.join(git_dir, 'jb-progress-2025/_extras')
fig_base_dir = os.path.join(git_dir, 'jb-progress-2025/figs')
tmp_dir = os.path.join(git_dir, 'jb-progress-2025/tmp')

# GitHub
sys.path.insert(0, os.path.join(git_dir, project_name))
from figures.fighelper import *
from main.train import *

# warnings, tqdm, & style
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
from rich.jupyter import print
%matplotlib inline
set_style()

## Setup

In [2]:
from base.helper import job_runner_script


def _cleanup(path, host=None):
    for f in os.listdir(path):
        cond = f.endswith('.txt')
        if host is not None:
            cond = cond and host in f
        if cond:
            os.remove(pjoin(path, f))


def _name(host, gpu_i, fit_i):
    return f"{host}-cuda{gpu_i}-fit{fit_i}"

In [3]:
save_dir = f'Dropbox/git/{project_name}/scripts'
save_dir = pjoin(os.environ['HOME'], save_dir)
os.makedirs(save_dir, exist_ok=True)

# delete existing job runners?
_cleanup(save_dir, None)

print(sorted(os.listdir(save_dir)))

[
    'cleanup_chkpts.sh',
    'cleanup_recursive.sh',
    'copyfits.sh',
    'fit_model.sh',
    'kill_screens.sh',
    'resume_fit.sh',
    'run_sessions.sh',
    'test_tqdm.py',
    'test_tqdm.sh'
]

## mach (```T=16```)

gaussian+relu

FashionMNIST, ```<grad|lin>```

done.

In [4]:
host = 'mach'
_cleanup(save_dir, host)

scripts = collections.defaultdict(list)
tot = 0

In [5]:
seeds = np.arange(10, 10 + 5)

model_type = 'gaussian'
latent_act = 'relu'

seq_len = 16
dataset = 'FashionMNIST'

seeds

array([10, 11, 12, 13, 14])

In [6]:
for seed in seeds:
    gpu_i = (seed // 12) * 2
    print(f"seed = {seed},\tgpu_i = {gpu_i}")

seed = 10,      gpu_i = 0

seed = 11,      gpu_i = 0

seed = 12,      gpu_i = 2

seed = 13,      gpu_i = 2

seed = 14,      gpu_i = 2

In [7]:
tot = 0

for seed in seeds:
    arg = [
        f"--seq_len {seq_len}",
        f"--kl_beta {float(seq_len)}",
        f"--latent_act '{latent_act}'" if latent_act else '',
        '--verbose',
    ]
    arg = ' '.join(filter(None, arg))

    # gpu_i = tot % torch.cuda.device_count()
    gpu_i = (seed // 12) * 2

    kws = dict(
        device=gpu_i,
        dataset=dataset,
        model=model_type,
        args=arg,
        seed=seed,
    )
    scripts[gpu_i].append(job_runner_script(**kws))
    tot += 1

In [8]:
print(tot)

5

In [9]:
scripts = dict(sorted(scripts.items()))
print({k: len(v) for k, v in scripts.items()})

{0: 2, 2: 3}

### Save

In [10]:
n_fits = 3

for gpu_i, scripts_list in scripts.items():
    scripts_divided = divide_list(scripts_list, n_fits)
    for fit_i, s in enumerate(scripts_divided):
        combined = ' && '.join(s)
        save_obj(
            obj=combined,
            file_name=_name(host, gpu_i, fit_i),
            save_dir=save_dir,
            mode='txt',
        )
        print(combined.replace('&& ', '&& \n'))

[PROGRESS] 'mach-cuda0-fit0.txt' saved at
/home/hadi/Dropbox/git/_IterativeVAE/scripts

./fit_model.sh '0' 'FashionMNIST' 'gaussian' --seed 10 --seq_len 16 --kl_beta 16.0 --latent_act 'relu' --verbose

[PROGRESS] 'mach-cuda0-fit1.txt' saved at
/home/hadi/Dropbox/git/_IterativeVAE/scripts

./fit_model.sh '0' 'FashionMNIST' 'gaussian' --seed 11 --seq_len 16 --kl_beta 16.0 --latent_act 'relu' --verbose

[PROGRESS] 'mach-cuda0-fit2.txt' saved at
/home/hadi/Dropbox/git/_IterativeVAE/scripts

...

./fit_model.sh '2' 'FashionMNIST' 'gaussian' --seed 12 --seq_len 16 --kl_beta 16.0 --latent_act 'relu' --verbose

[PROGRESS] 'mach-cuda2-fit1.txt' saved at
/home/hadi/Dropbox/git/_IterativeVAE/scripts

./fit_model.sh '2' 'FashionMNIST' 'gaussian' --seed 13 --seq_len 16 --kl_beta 16.0 --latent_act 'relu' --verbose

[PROGRESS] 'mach-cuda2-fit2.txt' saved at
/home/hadi/Dropbox/git/_IterativeVAE/scripts

./fit_model.sh '2' 'FashionMNIST' 'gaussian' --seed 14 --seq_len 16 --kl_beta 16.0 --latent_act 'relu' --verbose

Print one to check

In [11]:
print(combined.replace('&& ', '&& \n'))

./fit_model.sh '2' 'FashionMNIST' 'gaussian' --seed 14 --seq_len 16 --kl_beta 16.0 --latent_act 'relu' --verbose

In [12]:
scripts[0]

["./fit_model.sh '0' 'FashionMNIST' 'gaussian' --seed 10 --seq_len 16 --kl_beta 16.0 --latent_act 'relu' --verbose",
 "./fit_model.sh '0' 'FashionMNIST' 'gaussian' --seed 11 --seq_len 16 --kl_beta 16.0 --latent_act 'relu' --verbose"]

## mach (```T=64```)

gaussian

FashionMNIST, ```<grad|lin>```

doing now

In [4]:
host = 'mach'
_cleanup(save_dir, host)

scripts = collections.defaultdict(list)
tot = 0

In [5]:
seeds = np.arange(10, 10 + 5)

model_type = 'gaussian'
latent_act = None

seq_len = 64
dataset = 'FashionMNIST'

seeds

array([10, 11, 12, 13, 14])

In [6]:
for seed in seeds:
    # gpu_i = (seed // 12) * 2
    gpu_i = (tot + 1) % torch.cuda.device_count()
    print(f"seed = {seed},\tgpu_i = {gpu_i}")
    tot+=1

seed = 10,      gpu_i = 1

seed = 11,      gpu_i = 2

seed = 12,      gpu_i = 3

seed = 13,      gpu_i = 0

seed = 14,      gpu_i = 1

In [7]:
tot = 0

for seed in seeds:
    arg = [
        f"--seq_len {seq_len}",
        f"--kl_beta {float(seq_len)}",
        f"--latent_act '{latent_act}'" if latent_act else '',
        '--verbose',
    ]
    arg = ' '.join(filter(None, arg))

    # gpu_i = tot % torch.cuda.device_count()
    gpu_i = (tot + 1) % torch.cuda.device_count()

    kws = dict(
        device=gpu_i,
        dataset=dataset,
        model=model_type,
        args=arg,
        seed=seed,
    )
    scripts[gpu_i].append(job_runner_script(**kws))
    tot += 1

In [8]:
print(tot)

5

In [9]:
scripts = dict(sorted(scripts.items()))
print({k: len(v) for k, v in scripts.items()})

{0: 1, 1: 2, 2: 1, 3: 1}

### Save

In [10]:
n_fits = 2

for gpu_i, scripts_list in scripts.items():
    scripts_divided = divide_list(scripts_list, n_fits)
    for fit_i, s in enumerate(scripts_divided):
        combined = ' && '.join(s)
        save_obj(
            obj=combined,
            file_name=_name(host, gpu_i, fit_i),
            save_dir=save_dir,
            mode='txt',
        )
        print(combined.replace('&& ', '&& \n'))

[PROGRESS] 'mach-cuda0-fit0.txt' saved at
/home/hadi/Dropbox/git/_IterativeVAE/scripts

./fit_model.sh '0' 'FashionMNIST' 'gaussian' --seed 13 --seq_len 64 --kl_beta 64.0 --verbose

[PROGRESS] 'mach-cuda0-fit1.txt' saved at
/home/hadi/Dropbox/git/_IterativeVAE/scripts

[PROGRESS] 'mach-cuda1-fit0.txt' saved at
/home/hadi/Dropbox/git/_IterativeVAE/scripts

./fit_model.sh '1' 'FashionMNIST' 'gaussian' --seed 10 --seq_len 64 --kl_beta 64.0 --verbose

...

./fit_model.sh '2' 'FashionMNIST' 'gaussian' --seed 11 --seq_len 64 --kl_beta 64.0 --verbose

[PROGRESS] 'mach-cuda2-fit1.txt' saved at
/home/hadi/Dropbox/git/_IterativeVAE/scripts

[PROGRESS] 'mach-cuda3-fit0.txt' saved at
/home/hadi/Dropbox/git/_IterativeVAE/scripts

./fit_model.sh '3' 'FashionMNIST' 'gaussian' --seed 12 --seq_len 64 --kl_beta 64.0 --verbose

[PROGRESS] 'mach-cuda3-fit1.txt' saved at
/home/hadi/Dropbox/git/_IterativeVAE/scripts

Print one to check

In [11]:
print(combined.replace('&& ', '&& \n'))

In [12]:
scripts[0]

["./fit_model.sh '0' 'FashionMNIST' 'gaussian' --seed 13 --seq_len 64 --kl_beta 64.0 --verbose"]

## yoru (```T=128```)

poisson

FsahionMNIST, ```<grad|lin>```

TODO

In [4]:
host = 'yoru'
_cleanup(save_dir, host)

scripts = collections.defaultdict(list)
tot = 0

In [5]:
seeds = np.arange(10, 10 + 5)

model_type = 'poisson'
latent_act = None

seq_len = 128
beta = 128.0

dataset = 'EMNIST'

seeds

array([10, 11, 12, 13, 14])

In [6]:
tot = 0

for seed in seeds:
    arg = [
        f"--seq_len {seq_len}",
        f"--kl_beta {beta}",
        f"--latent_act '{latent_act}'" if latent_act else '',
        '--verbose',
    ]
    arg = ' '.join(filter(None, arg))
    gpu_i = tot % torch.cuda.device_count()
    kws = dict(
        device=gpu_i,
        dataset=dataset,
        model=model_type,
        args=arg,
        seed=seed,
    )
    scripts[gpu_i].append(job_runner_script(**kws))
    tot += 1

In [7]:
print(tot)

5

In [8]:
scripts = dict(scripts)
print({k: len(v) for k, v in scripts.items()})

{0: 2, 1: 1, 2: 1, 3: 1}

### Save

In [9]:
n_fits = 1

for gpu_i, scripts_list in scripts.items():
    scripts_divided = divide_list(scripts_list, n_fits)
    for fit_i, s in enumerate(scripts_divided):
        combined = ' && '.join(s)
        save_obj(
            obj=combined,
            file_name=_name(host, gpu_i, fit_i),
            save_dir=save_dir,
            mode='txt',
        )
        print(combined.replace('&& ', '&& \n'))

[PROGRESS] 'yoru-cuda0-fit0.txt' saved at
/home/hadi/Dropbox/git/_IterativeVAE/scripts

./fit_model.sh '0' 'EMNIST' 'poisson' --seed 10 --seq_len 128 --kl_beta 128.0 --verbose && 
./fit_model.sh '0' 'EMNIST' 'poisson' --seed 14 --seq_len 128 --kl_beta 128.0 --verbose

[PROGRESS] 'yoru-cuda1-fit0.txt' saved at
/home/hadi/Dropbox/git/_IterativeVAE/scripts

./fit_model.sh '1' 'EMNIST' 'poisson' --seed 11 --seq_len 128 --kl_beta 128.0 --verbose

[PROGRESS] 'yoru-cuda2-fit0.txt' saved at
/home/hadi/Dropbox/git/_IterativeVAE/scripts

./fit_model.sh '2' 'EMNIST' 'poisson' --seed 12 --seq_len 128 --kl_beta 128.0 --verbose

[PROGRESS] 'yoru-cuda3-fit0.txt' saved at
/home/hadi/Dropbox/git/_IterativeVAE/scripts

./fit_model.sh '3' 'EMNIST' 'poisson' --seed 13 --seq_len 128 --kl_beta 128.0 --verbose

Print one to check

In [10]:
print(combined.replace('&& ', '&& \n'))

./fit_model.sh '3' 'EMNIST' 'poisson' --seed 13 --seq_len 128 --kl_beta 128.0 --verbose